# Normalized Laptop Price Model

This notebook trains a separate model using normalized numeric features and categories aligned with the website form.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [2]:
# This works when the notebook is opened from either the repository root or model/.
ROOT = Path.cwd()
if not (ROOT / 'data set' / 'laptop_price.csv').exists():
    ROOT = ROOT / 'model'

DATA_PATH = ROOT / 'data set' / 'laptop_price.csv'
MODEL_PATH = ROOT / 'Website' / 'Model' / 'predictor_normalized.pickle'
REPORT_PATH = ROOT / 'normalized_model_report.txt'

data = pd.read_csv(DATA_PATH, encoding='latin-1')
print(f'Loaded {len(data)} rows from {DATA_PATH}')
data.head()

Loaded 1303 rows from d:\My Projects\Laptop-Price-Predictor\model\data set\laptop_price.csv


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


## Convert raw values to form-aligned features

RAM and weight become numeric. Rare companies and operating systems are grouped into the same categories offered by the form. Exact product names, laptop IDs, and exact CPU/GPU names are excluded because the form does not collect them.

In [3]:
FORM_COMPANIES = {'Acer', 'Apple', 'Asus', 'Dell', 'HP', 'Lenovo', 'MSI', 'Toshiba'}

def to_form_features(data):
    result = pd.DataFrame(index=data.index)
    result['Ram_GB'] = data['Ram'].str.replace('GB', '', regex=False).astype(float)
    result['Weight_KG'] = data['Weight'].str.replace('kg', '', regex=False).astype(float)

    result['Company'] = data['Company'].where(data['Company'].isin(FORM_COMPANIES), 'Other')
    result['TypeName'] = data['TypeName']

    result['OpSys'] = np.select(
        [
            data['OpSys'].str.startswith('Windows'),
            data['OpSys'].isin(['macOS', 'Mac OS X']),
            data['OpSys'].eq('Linux'),
        ],
        ['Windows', 'Mac', 'Linux'],
        default='Other',
    )

    result['CPU'] = np.select(
        [
            data['Cpu'].str.contains('Intel Core i3', regex=False),
            data['Cpu'].str.contains('Intel Core i5', regex=False),
            data['Cpu'].str.contains('Intel Core i7', regex=False),
            data['Cpu'].str.startswith('AMD'),
        ],
        ['Intel Core i3', 'Intel Core i5', 'Intel Core i7', 'AMD'],
        default='Other',
    )

    result['GPU'] = np.select(
        [data['Gpu'].str.startswith('Nvidia'), data['Gpu'].str.startswith('AMD')],
        ['Nvidia', 'AMD'],
        default='Intel',
    )

    result['Touchscreen'] = data['ScreenResolution'].str.contains(
        'Touchscreen', regex=False
    ).map({True: 'Yes', False: 'No'})
    result['IPS'] = data['ScreenResolution'].str.contains(
        'IPS', regex=False
    ).map({True: 'Yes', False: 'No'})
    return result

features = to_form_features(data)
target = data['Price_euros']
features.head()

,Ram_GB,Weight_KG,Company,TypeName,OpSys,CPU,GPU,Touchscreen,IPS
0,8.0,1.37,Apple,Ultrabook,Mac,Intel Core i5,Intel,No,Yes
1,8.0,1.34,Apple,Ultrabook,Mac,Intel Core i5,Intel,No,No
2,8.0,1.86,HP,Notebook,Other,Intel Core i5,Intel,No,No
3,16.0,1.83,Apple,Ultrabook,Mac,Intel Core i7,AMD,No,Yes
4,8.0,1.37,Apple,Ultrabook,Mac,Intel Core i5,Intel,No,Yes


In [4]:
numeric_features = ['Ram_GB', 'Weight_KG']
categorical_features = ['Company', 'TypeName', 'OpSys', 'CPU', 'GPU', 'Touchscreen', 'IPS']

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', MinMaxScaler(), numeric_features),
        (
            'categorical',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            categorical_features,
        ),
    ]
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestRegressor(
                n_estimators=500,
                criterion='absolute_error',
                min_samples_leaf=2,
                max_features=0.8,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)
pipeline.fit(x_train, y_train)
predictions = pipeline.predict(x_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'MAE:  {mae:.2f} EUR')
print(f'RMSE: {rmse:.2f} EUR')
print(f'R²:   {r2:.4f}')

MAE:  223.13 EUR
RMSE: 374.51 EUR
R²:   0.7363


In [6]:
cv_mae = -cross_val_score(
    pipeline,
    features,
    target,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=1,
)

print('5-fold MAE values:', [round(value, 2) for value in cv_mae])
print(f'Mean cross-validation MAE: {cv_mae.mean():.2f} EUR')
print(f'Cross-validation MAE std: {cv_mae.std():.2f} EUR')

5-fold MAE values: [np.float64(223.04), np.float64(194.03), np.float64(205.74), np.float64(232.06), np.float64(233.86)]
Mean cross-validation MAE: 217.75 EUR
Cross-validation MAE std: 15.49 EUR


In [7]:
# Save the separate model. This does not replace the existing predictor.pickle.
MODEL_PATH.write_bytes(pickle.dumps(pipeline, protocol=pickle.HIGHEST_PROTOCOL))
print(f'Saved normalized model to {MODEL_PATH}')

Saved normalized model to d:\My Projects\Laptop-Price-Predictor\model\Website\Model\predictor_normalized.pickle


In [8]:
# Example prediction using the same normalized feature names expected by the pipeline.
example = pd.DataFrame([{
    'Ram_GB': 16,
    'Weight_KG': 2.0,
    'Company': 'Lenovo',
    'TypeName': 'Gaming',
    'OpSys': 'Windows',
    'CPU': 'Intel Core i7',
    'GPU': 'Nvidia',
    'Touchscreen': 'No',
    'IPS': 'Yes',
}])

estimated_eur = pipeline.predict(example)[0]
print(f'Example estimate: €{estimated_eur:,.2f}')

Example estimate: €1,910.42
